# 06 · Full baseline run

The full baseline answers every item of `run_benchmarks` x `run_splits` at each loop count in `R_grid`, with the same code path as `scripts/mvp.py full`. Items are sharded (`items[i::n]`), one shard per GPU group, and each shard writes its own resumable JSONL. This notebook first checks batched generation against batch 1, then runs the full run only when `RUN_FULL` is True, then summarises the result files. The test split stays off until stage S12.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import socket

In [ ]:
from adaloop import api  # importing api first; setup_notebook puts the repo root on sys.path

ROOT, cfg = api.setup_notebook("configs/full_baseline.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
# torch is imported here because the GPU list needs it; the other third-party imports come after this cell
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("CUDA_VISIBLE_DEVICES", os.environ.get("CUDA_VISIBLE_DEVICES", "unset (all GPUs visible)"))
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"torch GPU {i}: {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")
else:
    print("no CUDA GPU visible: the model steps will fail, the rest of the notebook can still be read")
print("gpu_groups", cfg["gpu_groups"], "| only these physical GPUs (0-3 on Inara) may be used")

In [ ]:
import pandas as pd

In [ ]:
# Settings: the full run is off by default because it is long
N_CHECK = 8  # dev items for the batch check: half GQA, half TextVQA
R_CHECK = 3  # loop passes for the batch check
RUN_FULL = False  # step 2: True runs shard SHARD in this kernel
SHARD = (0, 2)  # step 2: (i, n) run by this kernel; the terminal commands in step 2 do the rest

### Step-1:- Batch check: batch 1 vs batched generation

Batched generation pads the prompts to one length. Padding and batched GPU kernels can change the numerics, and a near-tied
choice can flip, so `batch_size` may give a different answer than batch 1. This step runs the same dev items at batch 1 and at
`batch_size` from the config, then compares the predictions. A mismatch must be reported, not ignored: the table shows which
items differ, and the full run should not start on batched rows until that is decided.

In [ ]:
print("[STAGE] load model")
vlm = api.load_model(cfg)

In [ ]:
print("[STAGE] batch check inputs")
# half GQA, half TextVQA, so one batch mixes both benchmarks
check_items = (
    api.load_gqa(cfg, "dev", limit=N_CHECK // 2, root=ROOT)
    + api.load_textvqa(cfg, "dev", limit=N_CHECK // 2, root=ROOT)
)
assert len(check_items) == N_CHECK, len(check_items)
print(f"{len(check_items)} items | batch sizes to compare: 1 and {cfg['batch_size']}")

In [ ]:
BATCH = cfg["batch_size"]
# one file per batch size; the name ends with the hash of the same run config the full run uses
check_rows = {}
for batch_size in (1, BATCH):
    run_cfg = api.full_run_config(cfg, batch_size)
    out = ROOT / cfg["out_dir"] / f"check_dev_n{N_CHECK}_R{R_CHECK}_b{batch_size}_{api.config_hash(run_cfg)}.jsonl"
    print(f"[STAGE] batch {batch_size} -> {out.name}")
    out.unlink(missing_ok=True)  # always regenerate: rows kept from older code would hide a mismatch
    api.run_evaluation(vlm, check_items, run_cfg, R_CHECK, out, resume=True, batch_size=batch_size)
    check_rows[batch_size] = {row["item_id"]: row for row in api.read_results(out)}

In [ ]:
check_table = pd.DataFrame([
    {
        "benchmark": item.benchmark,
        "item_id": item.item_id,
        "first_answer": item.answers[0],
        "pred_b1": check_rows[1][item.item_id]["pred"],
        f"pred_b{BATCH}": check_rows[BATCH][item.item_id]["pred"],
        "raw_b1": check_rows[1][item.item_id]["raw_output"],
        f"raw_b{BATCH}": check_rows[BATCH][item.item_id]["raw_output"],
    }
    for item in check_items
])
check_table["equal"] = check_table["pred_b1"] == check_table[f"pred_b{BATCH}"]
check_table

In [ ]:
mismatch = check_table[~check_table["equal"]]
print(f"mismatches: {len(mismatch)} of {len(check_table)}")
mismatch

In [ ]:
assert mismatch.empty, f"batch {BATCH} changed {len(mismatch)} prediction(s); see the table above before the full run"
print(f"✓ batch check: batch 1 and batch {BATCH} give the same predictions on {N_CHECK} items")

### Step-2:- Full run, one shard per GPU group

`RUN_FULL` runs shard `SHARD` in this kernel, on the GPUs this kernel can see, through the same `api.run_full` the CLI calls.
Shard files carry the shard in their name, so the kernel and the CLI write the same files.

For the two-shard run, use the terminal on Inara, one shard per GPU group:

```
python scripts/mvp.py full --shard 0/2
python scripts/mvp.py full --shard 1/2
```

Each command sets `CUDA_VISIBLE_DEVICES` to its group in `gpu_groups` (GPUs 0-1 and 2-3). At most 2 model copies fit on
GPUs 0-3, and this kernel may still hold GPUs 0 and 1 from step 1, so free them first (restart the kernel, or run
`del vlm; api.unload_model()`) before a terminal run that uses group 0. Both runs are resumable: rerun the same command after an
interruption and finished items are skipped. Do not set `RUN_FULL` for a shard that a terminal is already running.

In [ ]:
if RUN_FULL:
    print(f"[STAGE] full run, shard {SHARD[0]}/{SHARD[1]}")
    api.run_full(vlm, cfg, ROOT, shard=SHARD)
else:
    print("skipped: RUN_FULL is False")

### Step-3:- Summary per benchmark x R

Accuracy, parse rate and seconds per item, read from every shard file of this run config (`batch_size` from the config).
Run splits are pooled, each item counts once. For batched rows, s/item is batch time divided by batch size, measured on
2x A40 with balanced placement. The table is empty until result files exist.

In [ ]:
summary = api.summarize_full(cfg, ROOT)
full_table = (
    pd.DataFrame(summary)
    if summary
    else f"no result files yet for batch_size {cfg['batch_size']} in {cfg['out_dir']}"
)
full_table

### Step-4:- Cost model (stage S6)

Reserved: the FLOPs and latency cost-model cells are added here at stage S6. Nothing is written yet.